In [4]:
%pip install -q sentence-transformers faiss-cpu pandas numpy tqdm torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 106.7 MB/s eta 0:00:0000:0100:01


In [5]:
import pandas as pd
import numpy as np
import faiss
import torch

from sentence_transformers import SentenceTransformer

In [6]:
print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")

CUDA Available: True
GPU: Tesla T4


In [12]:
from google.colab import drive

drive.mount('/content/drive')


Mounted at /content/drive


In [19]:
PROJECT_DIR = "/content/drive/MyDrive/ROUND1"
DATA_DIR = os.path.join(PROJECT_DIR, "data")

print(DATA_DIR)

/content/drive/MyDrive/ROUND1/data


In [20]:
train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))

train["Reviews"] = train["Reviews"].fillna("")
test["Reviews"] = test["Reviews"].fillna("")

print("Train Shape:", train.shape)
print("Test Shape :", test.shape)

train.head()

Train Shape: (109776, 3)
Test Shape : (10977, 2)


,Index,Reviews,Course
0,0,React Native Mobile Development exceeded my ex...,React Native Mobile Development
1,1,I want to share my detailed thoughts on Deep L...,Deep Learning with TensorFlow
2,2,I recently completed Unsupervised Learning Tec...,Unsupervised Learning Techniques
3,3,I enrolled in Python for Absolute Beginners ho...,Python for Absolute Beginners
4,4,Finally completed Git and GitHub Mastery after...,Git and GitHub Mastery


In [21]:
print("CUDA Available :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))

CUDA Available : True
GPU : Tesla T4


In [22]:
model = SentenceTransformer(
    "BAAI/bge-base-en-v1.5",
    device="cuda" if torch.cuda.is_available() else "cpu"
)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [23]:
train_embeddings = model.encode(
    train["Reviews"].tolist(),
    batch_size=256,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=True
)

print(train_embeddings.shape)

Batches:   0%|          | 0/429 [00:00<?, ?it/s]

(109776, 768)


In [24]:
test_embeddings = model.encode(
    test["Reviews"].tolist(),
    batch_size=256,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=True
)

print(test_embeddings.shape)

Batches:   0%|          | 0/43 [00:00<?, ?it/s]

(10977, 768)


In [25]:
np.save(os.path.join(DATA_DIR, "train_embeddings.npy"), train_embeddings)
np.save(os.path.join(DATA_DIR, "test_embeddings.npy"), test_embeddings)

print("Embeddings saved.")

Embeddings saved.


In [26]:
dimension = train_embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(train_embeddings.astype(np.float32))

print("Indexed:", index.ntotal)

Indexed: 109776
